# Write new keys into a batched KV cache

A JAX KV cache is a fixed-size array (shapes must be static for `jit`), and writing into it is a functional update that returns a new array. In a batch, every sequence is at a **different** position, so each row gets its own write offset.

Write `update_cache(k_cache, v_cache, k_new, v_new, pos)`:

- `k_cache`, `v_cache`: `(B, S_max, H, d)` float32.
- `k_new`, `v_new`: `(B, T_new, H, d)` float32, the keys and values of the new tokens.
- `pos`: `(B,)` int, the current length of each sequence. Row `b`'s new entries go to positions `pos[b] .. pos[b] + T_new - 1`. It is guaranteed that `pos[b] + T_new <= S_max`.

Return `(k_cache, v_cache, lengths, valid)`:

- the updated caches (everything outside the written slots unchanged);
- `lengths = pos + T_new`, `(B,)` int;
- `valid`: `(B, S_max)` bool, `True` at positions `< lengths[b]`.

Use `jax.lax.dynamic_update_slice` with `jax.vmap` over the batch. No Python loop over `B`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def update_cache(k_cache, v_cache, k_new, v_new, pos):
    """Return (k_cache, v_cache, lengths (B,), valid (B, S_max) bool)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0]], [[0.0]], [[0.0]], [[0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0]], [[0.0]], [[0.0]], [[0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0]]]], "dtype": "float64"}, {"__nd__": [[[[2.0]]]], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.0]], [[0.0]], [[1.0]], [[0.0]]]], "dtype": "float32"}, {"__nd__": [[[[0.0]], [[0.0]], [[2.0]], [[0.0]]]], "dtype": "float32"}, {"__nd__": [3], "dtype": "int32"}, {"__nd__": [[true, true, true, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]]], [[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]]], [[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0]], [[1.0, 1.0]]], [[[1.0, 1.0]], [[1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[2.0, 2.0]], [[2.0, 2.0]]], [[[2.0, 2.0]], [[2.0, 2.0]]]], "dtype": "float64"}, {"__nd__": [0, 1], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[1.0, 1.0]], [[1.0, 1.0]], [[0.0, 0.0]]], [[[0.0, 0.0]], [[1.0, 1.0]], [[1.0, 1.0]]]], "dtype": "float32"}, {"__nd__": [[[[2.0, 2.0]], [[2.0, 2.0]], [[0.0, 0.0]]], [[[0.0, 0.0]], [[2.0, 2.0]], [[2.0, 2.0]]]], "dtype": "float32"}, {"__nd__": [2, 3], "dtype": "int32"}, {"__nd__": [[true, true, false], [true, true, true]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[[[-1.9250644981038072, -3.018795230612003, 1.1544631465797832, 1.5627819063534067], [1.329517371305854, -0.4295924953879639, -0.07288832141889304, -0.29154696136252883]], [[1.368242680287551, 0.8719673488267038, 0.058377719807040734, 0.7081236283128302], [1.36030603397752, 0.3745250527216613, -1.0855167946423785, -0.07479491444513282]], [[-0.426656550198213, -1.5918472886161878, -2.0458564600786957, 0.14871256099989164], [-1.045382122238908, 0.41432150070543916, -0.8002148300104912, 0.8030278646747248]], [[0.885075041426785, 0.3822621141833038, -1.1992640930491332, -2.0412527590777985], [2.141861457996631, -0.05345460228944404, 2.0492663308238317, -0.23760244649637505]], [[-1.1791143385328935, 0.00381842748735191, -1.2333371670837567, 0.04694664869575337], [1.645894371231227, 0.9012759138878724, 3.100969551187694, 0.3708480695387392]], [[0.3482981312485577, -0.9027846477038957, -0.9296628437469088, 0.27654000624419384], [-0.8878052343125458, -0.31328904474023855, -0.30420308225350584, -1.1553833316193558]], [[2.3403069890702035, -1.9401215592164645, 0.15619020660676766, 1.0003005760005261], [1.13755435573667, -0.4147559601689877, -1.0318733186058924, 2.478020237174974]], [[2.2227452991236625, 0.18007624885742057, 0.05161167546249193, 0.13230574445475254], [-1.1123575085393496, -0.01576889465045249, 1.1388359861283301, 0.7828972849722188]]], [[[-0.9352105606807994, 0.816469979945203, -0.8215180112421969, -1.641545149753898], [0.3932832934745938, 0.6868793168356786, 1.5288996245701283, -0.8398689685333504]], [[1.1063131384159979, 0.5854940180914177, 0.2876183510971611, -0.40821130037834447], [-0.228812880699083, 0.5337960431353989, 0.3628524047067793, -1.2089141540333015]], [[-0.5890422527075397, -0.08100093739124811, 1.0097336223493647, 0.48744606959933584], [-0.02661276366775354, -0.2527425147611129, 1.6320775839813169, -0.46711248604948313]], [[-0.7922513500841074, -0.4597440799639441, 0.3681197966163168, 0.006028550825019775], [-0.558305124694066, -0.47917383720977924, 0.5502155334835439, 0.49592969197268166]], [[0.3340832884901645, -0.9555959984772731, 0.01522022771414626, 1.2532379789139636], [-0.735900637902722, 1.7466062722291409, 0.24810301459803077, 0.9904992738405]], [[-1.0230518912095272, 1.4328297455751997, 0.09864330944737564, 0.45830148252383296], [0.7669404545304286, 1.679651223150558, -0.1091099376213063, 0.706242504102666]], [[0.2788190854626221, -1.462396541932785, -1.1024642154713509, -1.3511691965950066], [-0.058968910790412915, 0.32637618017427145, 0.6057781913426709, 1.100045566577106]], [[0.3619022958593807, -1.1884839010043575, 1.8788355367218468, 0.2582693128871191], [-0.5303995787367197, -1.5528099975648566, 0.6603080206024908, -1.4772267630150608]]], [[[1.2845638520138312, 0.30328371320507, -1.7464921841302476, -0.3994212709668019], [1.9062545848276309, -1.8983524859691752, -0.51701828470523, -2.1488657841532075]], [[0.13091409264377976, 1.8677707887005994, -0.057285451728603734, -0.15617426916330202], [-0.35388300085504293, -0.3120102149057473, -0.3912187698204805, 0.6617662796065238]], [[1.0462930927558534, -2.4450450490303193, -2.737119094352514, 1.0037246850315327], [1.0023787245290476, -1.5358227785193723, -0.35292724880360987, 0.29624776715323653]], [[0.6734833836993913, 0.07506591848575707, 0.2549741005327368, -0.2861217483707138], [0.7360549193688412, -0.08423272286890403, -0.31761802872938766, -0.4105796502525977]], [[-0.7779058109393737, 0.638990576373326, 0.2525834565710265, 0.20735221862054537], [-0.37040971814785745, 0.6900778731821201, 0.2319338690055988, -1.2230539607852084]], [[-2.59774732148878, -0.5711171546108783, -0.16081856945479495, 2.0548609247879894], [1.3692423554885849, 0.7697395356377859, 2.423165798863212, -0.5912988795411409]], [[0.15528180431941552, 0.8010423099769229, -0.04448479313778214, -0.21356634240823932], [0.40054298015273765, -0.18787722501741055, 1.155819801078235, -2.015411542036465]], [[0.44578619232370015, 1.7056273598421068, -0.6335939899657264, -0.8517744296790382], [0.30824000990593353, -0.03977993018535824, 0.47559969379188544, -0.5249200761328455]]]], "dtype": "float64"}, {"__nd__": [[[[0.24456725843801677, 0.6970760314534526, -2.347075109013072, -0.058631556468542734], [0.48843748558659256, -1.2452450470815248, -1.3314513625855637, 0.8650566941317135]], [[-0.724724176463259, 0.26648493811683194, 0.7807874242206715, 0.7916991977831074], [-1.6272415466294665, 0.19036042829577668, -0.6877039818681961, -2.158348211287]], [[0.2935103670662315, 0.7787189680750223, -0.4425537072775435, 0.5656741222402984], [0.31415024533624125, 0.18922552223935057, -1.8850820717694738, 0.5951980023335661]], [[-1.4057016935079034, -1.5918246710102832, -0.7466896273348441, 1.217003886841809], [0.5833958181535286, 1.4014225838740348, -0.3323407262082055, 0.5346115198496569]], [[-0.43795015104655466, -0.5788765631806206, 0.4836615009682777, -0.07290279243796706], [-1.879171994656977, 0.5354866440868556, -0.5936547767371834, 1.9017581448036374]], [[-0.008072117714551588, 0.24594662382597748, -1.525998788925307, 1.0762625027988288], [0.6398508891323629, -0.9942654873791151, -0.390434221287234, -1.318909158521955]], [[1.908765662378363, -1.2832539368078935, 1.1063120570497236, -1.0455814101086434], [2.419752157187113, 1.648923211325512, 0.9417507039722247, -0.16746220109197693]], [[-0.5548925728439811, 0.6262484573781733, 0.3411556522116535, -1.0827108504334986], [1.1144368640266291, 0.4452178712737237, -0.41360086453575523, -0.09797931424799583]]], [[[0.5242231048532486, 0.6345536404316853, 0.08814316595660761, 0.1819082368431576], [-1.716258442569262, 1.8939036030158636, -0.05531063390701612, -1.0705000048022402]], [[0.7093295010973732, -1.5071676909556369, -0.14894239106789986, -2.7906950995459976], [-0.30915150526404356, -0.9554714820156793, -0.08046852732288907, 1.13723151206313]], [[1.2788920823440073, -0.4368875056210777, 1.4522804414520782, 1.7912774815128305], [1.1223342904465243, -1.097078417979293, -0.5588127996638321, 1.0417467863061558]], [[-1.5593168539368225, 0.9865560967416638, 0.04690119220223484, 2.350537095035514], [0.02740931336623944, 0.7426133191768496, -0.4170170954909958, -1.2715630191674798]], [[0.09463827591201271, -0.7754999395279703, 0.8798240850744081, 0.880807711338415], [0.31081827612327156, -0.17446195312638504, 0.09478274736645932, -0.567684731042421]], [[-0.13135690062196423, 0.21175959150210324, 0.43407984739854316, -0.31295075449251647], [0.17887288825750647, -1.0690045958215588, 0.17018493405980037, 1.945476589077128]], [[0.28925032446005233, -0.13151030486569837, -0.36531855111617223, -1.297021124165389], [0.26860747378981514, 0.6399688418017188, 0.22194510850991253, -0.5672583836813085]], [[0.03539913278082404, -0.4224098751538277, 1.1908860076169183, -0.4520434071009424], [0.8994444266573255, -0.5907788314230251, -0.890603710582627, 0.004515209675520739]]], [[[0.0692334573904048, -0.5345362968340817, 0.9584209483272421, -1.4136043327507293], [-0.19935768593619677, 0.013283593106119686, -0.7708669623313305, 0.26903686308358765]], [[-1.4388018389981858, -1.468536492236935, 0.19035297925711997, 0.1496100596166225], [1.119252600985713, 0.40365307161698594, 0.131427113797494, 0.207612630657573]], [[0.457600974595075, -1.2979028362877805, 0.5205759532527726, -0.15516480714178663], [0.5437314836072298, -1.4562227527920566, 0.9819552390671857, 1.3184523070823089]], [[0.40099471974902967, 2.307280632329832, -1.4632887116526498, -0.5823121191782743], [-0.03970367149831715, 0.6114435098670433, 0.10920989946731118, 0.37717166731269397]], [[0.6239346978883292, -0.5656505134431934, 1.2622781529231444, 1.2679702943702016], [-0.5198431799232827, -0.810904746823556, -0.5206465711648607, -0.9305330900876715]], [[0.1554746883002646, -0.08862700397865972, 0.521090817554083, -1.2456380971789196], [-0.07353504626330236, 0.002084217909100542, 0.3440546238823279, 0.37933792314920856]], [[0.3778650201536064, 0.03235280672230607, -1.210524006722515, -0.19850019467885566], [1.6827727168947721, 0.9493145840776002, 0.6939357842180994, -1.59958985968117]], [[0.581937608992179, 0.8723009582897712, -0.1204730158676558, -1.5725696827285183], [1.2947046277397678, -0.6960249237660462, 0.13735040109213806, 0.7919547758130975]]]], "dtype": "float64"}, {"__nd__": [[[[0.9513242894428062, 0.6818785881184019, -2.3222386938676625, -0.2222540847458842], [-1.0459213729167678, -0.05160154402665502, -0.8098304252765397, 1.7847641525661955]]], [[[-1.5405814524679777, 1.1062534620246982, -0.1595807495134806, 0.46729988949539125], [-0.3075519299413692, -0.41931660019249345, -0.8134379383790267, 1.353230509252111]]], [[[0.15805102012401556, -0.21879845331288944, 0.41459868670511046, 0.2384216620091448], [-0.34260328372320903, 0.3410099968386834, 0.5241065968512495, 1.7579912142847056]]]], "dtype": "float64"}, {"__nd__": [[[[-1.0559338804947986, -2.046307585348845, -0.4155239305608111, 1.8881232865180282], [0.6159828790572841, -0.06551526161067146, -0.3544117823257437, -1.4358610097923452]]], [[[1.4055260040405981, 1.0976020466103653, 1.7199712612285494, -0.9792960620557796], [1.4317558761309397, 0.946196216023573, 0.6901294113116985, -1.0152737053267125]]], [[[0.0921759464316291, -0.613426697726726, 0.10111320789254899, -1.695191245116964], [-1.0693092277318719, -0.42841562340384065, -1.9895712441829052, 0.27841259463859963]]]], "dtype": "float64"}, {"__nd__": [0, 5, 7], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.9513242840766907, 0.6818785667419434, -2.3222386837005615, -0.22225408256053925], [-1.0459213256835938, -0.05160154402256012, -0.8098304271697998, 1.7847641706466675]], [[1.368242621421814, 0.8719673752784729, 0.058377720415592194, 0.7081236243247986], [1.3603060245513916, 0.3745250403881073, -1.0855168104171753, -0.07479491084814072]], [[-0.4266565442085266, -1.59184730052948, -2.045856475830078, 0.14871256053447723], [-1.0453821420669556, 0.41432151198387146, -0.8002148270606995, 0.803027868270874]], [[0.885075032711029, 0.38226211071014404, -1.1992640495300293, -2.041252851486206], [2.1418614387512207, -0.053454603999853134, 2.0492663383483887, -0.23760244250297546]], [[-1.1791143417358398, 0.0038184274453669786, -1.233337163925171, 0.04694664850831032], [1.6458944082260132, 0.9012759327888489, 3.1009695529937744, 0.37084805965423584]], [[0.3482981324195862, -0.9027846455574036, -0.929662823677063, 0.27654001116752625], [-0.8878052234649658, -0.3132890462875366, -0.3042030930519104, -1.1553833484649658]], [[2.3403069972991943, -1.9401215314865112, 0.1561902016401291, 1.0003005266189575], [1.137554407119751, -0.4147559702396393, -1.031873345375061, 2.478020191192627]], [[2.222745180130005, 0.1800762414932251, 0.051611676812171936, 0.13230574131011963], [-1.112357497215271, -0.01576889492571354, 1.1388360261917114, 0.7828972935676575]]], [[[-0.9352105855941772, 0.8164699673652649, -0.8215180039405823, -1.6415451765060425], [0.39328330755233765, 0.6868793368339539, 1.5288996696472168, -0.83986896276474]], [[1.1063131093978882, 0.5854940414428711, 0.28761833906173706, -0.40821129083633423], [-0.22881287336349487, 0.5337960720062256, 0.36285239458084106, -1.2089141607284546]], [[-0.5890422463417053, -0.08100093901157379, 1.0097336769104004, 0.4874460697174072], [-0.026612764224410057, -0.2527425289154053, 1.6320775747299194, -0.4671124815940857]], [[-0.7922513484954834, -0.45974406599998474, 0.3681198060512543, 0.006028550677001476], [-0.5583051443099976, -0.47917383909225464, 0.5502155423164368, 0.49592968821525574]], [[0.33408328890800476, -0.9555959701538086, 0.015220227651298046, 1.2532379627227783], [-0.7359006404876709, 1.7466062307357788, 0.24810300767421722, 0.9904992580413818]], [[-1.540581464767456, 1.1062535047531128, -0.1595807522535324, 0.4672998785972595], [-0.30755192041397095, -0.4193165898323059, -0.8134379386901855, 1.3532304763793945]], [[0.27881908416748047, -1.462396502494812, -1.102464199066162, -1.351169228553772], [-0.058968909084796906, 0.32637616991996765, 0.6057782173156738, 1.1000455617904663]], [[0.36190229654312134, -1.1884839534759521, 1.8788355588912964, 0.2582693099975586], [-0.5303995609283447, -1.5528099536895752, 0.6603080034255981, -1.477226734161377]]], [[[1.2845638990402222, 0.3032837212085724, -1.7464921474456787, -0.3994212746620178], [1.906254529953003, -1.8983525037765503, -0.5170182585716248, -2.1488656997680664]], [[0.1309140920639038, 1.867770791053772, -0.05728545039892197, -0.15617427229881287], [-0.3538829982280731, -0.3120102286338806, -0.39121878147125244, 0.6617662906646729]], [[1.0462931394577026, -2.445044994354248, -2.737119197845459, 1.0037246942520142], [1.0023787021636963, -1.5358227491378784, -0.35292723774909973, 0.29624778032302856]], [[0.6734833717346191, 0.07506591826677322, 0.2549740970134735, -0.28612175583839417], [0.7360548973083496, -0.08423272520303726, -0.3176180422306061, -0.410579651594162]], [[-0.7779058218002319, 0.638990581035614, 0.25258344411849976, 0.20735222101211548], [-0.3704097270965576, 0.6900779008865356, 0.2319338619709015, -1.2230539321899414]], [[-2.597747325897217, -0.5711171627044678, -0.16081857681274414, 2.054860830307007], [1.3692423105239868, 0.7697395086288452, 2.423165798187256, -0.5912988781929016]], [[0.15528179705142975, 0.8010423183441162, -0.04448479413986206, -0.2135663479566574], [0.4005429744720459, -0.18787722289562225, 1.1558197736740112, -2.015411615371704]], [[0.1580510139465332, -0.2187984585762024, 0.414598673582077, 0.23842166364192963], [-0.34260329604148865, 0.3410100042819977, 0.5241066217422485, 1.7579911947250366]]]], "dtype": "float32"}, {"__nd__": [[[[-1.0559338331222534, -2.0463075637817383, -0.4155239164829254, 1.8881232738494873], [0.6159828901290894, -0.06551526486873627, -0.354411780834198, -1.4358609914779663]], [[-0.7247241735458374, 0.26648494601249695, 0.7807874083518982, 0.7916991710662842], [-1.6272414922714233, 0.190360426902771, -0.6877039670944214, -2.158348321914673]], [[0.293510377407074, 0.7787189483642578, -0.44255369901657104, 0.5656741261482239], [0.31415024399757385, 0.18922552466392517, -1.8850821256637573, 0.5951979756355286]], [[-1.4057016372680664, -1.5918246507644653, -0.7466896176338196, 1.2170039415359497], [0.5833958387374878, 1.4014226198196411, -0.33234071731567383, 0.5346115231513977]], [[-0.43795016407966614, -0.5788765549659729, 0.4836615025997162, -0.07290279120206833], [-1.8791719675064087, 0.53548663854599, -0.5936547517776489, 1.9017581939697266]], [[-0.00807211734354496, 0.24594663083553314, -1.525998830795288, 1.0762624740600586], [0.639850914478302, -0.9942654967308044, -0.39043423533439636, -1.3189091682434082]], [[1.9087656736373901, -1.2832539081573486, 1.1063120365142822, -1.0455814599990845], [2.4197521209716797, 1.6489231586456299, 0.941750705242157, -0.16746219992637634]], [[-0.5548925995826721, 0.6262484788894653, 0.34115564823150635, -1.082710862159729], [1.1144368648529053, 0.44521787762641907, -0.4136008620262146, -0.09797931462526321]]], [[[0.5242230892181396, 0.6345536112785339, 0.08814316242933273, 0.1819082349538803], [-1.7162584066390991, 1.8939036130905151, -0.05531063303351402, -1.0705000162124634]], [[0.7093294858932495, -1.5071676969528198, -0.14894239604473114, -2.7906951904296875], [-0.3091515004634857, -0.9554714560508728, -0.08046852797269821, 1.137231469154358]], [[1.2788920402526855, -0.4368875026702881, 1.4522804021835327, 1.791277527809143], [1.1223342418670654, -1.0970784425735474, -0.5588127970695496, 1.041746735572815]], [[-1.559316873550415, 0.9865561127662659, 0.046901192516088486, 2.350537061691284], [0.02740931324660778, 0.7426133155822754, -0.4170171022415161, -1.2715630531311035]], [[0.09463827311992645, -0.7754999399185181, 0.8798241019248962, 0.8808076977729797], [0.31081828474998474, -0.1744619458913803, 0.0947827473282814, -0.5676847100257874]], [[1.405526041984558, 1.0976020097732544, 1.7199712991714478, -0.979296088218689], [1.4317559003829956, 0.9461961984634399, 0.6901293992996216, -1.0152736902236938]], [[0.28925031423568726, -0.13151030242443085, -0.36531853675842285, -1.2970211505889893], [0.2686074674129486, 0.6399688124656677, 0.22194510698318481, -0.567258358001709]], [[0.03539913147687912, -0.42240986227989197, 1.1908860206604004, -0.45204341411590576], [0.8994444012641907, -0.5907788276672363, -0.8906037211418152, 0.004515209700912237]]], [[[0.06923345476388931, -0.5345363020896912, 0.9584209322929382, -1.4136043787002563], [-0.19935768842697144, 0.013283592648804188, -0.7708669900894165, 0.2690368592739105]], [[-1.438801884651184, -1.4685364961624146, 0.19035297632217407, 0.14961005747318268], [1.1192525625228882, 0.403653085231781, 0.1314271092414856, 0.20761263370513916]], [[0.45760098099708557, -1.2979028224945068, 0.5205759406089783, -0.15516480803489685], [0.5437315106391907, -1.4562227725982666, 0.9819552302360535, 1.3184523582458496]], [[0.4009947180747986, 2.3072805404663086, -1.46328866481781, -0.5823121070861816], [-0.039703670889139175, 0.6114435195922852, 0.10920990258455276, 0.37717166543006897]], [[0.6239346861839294, -0.5656505227088928, 1.2622781991958618, 1.267970323562622], [-0.5198431611061096, -0.8109047412872314, -0.5206465721130371, -0.9305331110954285]], [[0.1554746925830841, -0.08862700313329697, 0.5210908055305481, -1.245638132095337], [-0.07353504747152328, 0.0020842179656028748, 0.34405460953712463, 0.37933793663978577]], [[0.3778650164604187, 0.03235280513763428, -1.2105239629745483, -0.19850020110607147], [1.6827727556228638, 0.9493145942687988, 0.6939358115196228, -1.5995898246765137]], [[0.09217594563961029, -0.613426685333252, 0.1011132076382637, -1.6951912641525269], [-1.0693092346191406, -0.4284156262874603, -1.989571213722229, 0.2784125804901123]]]], "dtype": "float32"}, {"__nd__": [1, 6, 8], "dtype": "int32"}, {"__nd__": [[true, false, false, false, false, false, false, false], [true, true, true, true, true, true, false, false], [true, true, true, true, true, true, true, true]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[[[0.7516971519498737, -0.4321773310219548, -0.1833874388864998]], [[-0.07590463582131651, 1.0735796326888032, 0.7674901324922221]], [[0.9898729736268069, 0.5962811993334852, 0.28782784008979073]], [[0.7464880712431422, 1.3027398137727764, 2.495110505568125]], [[1.0304616736222139, -0.4316544120373864, -0.47887917136857105]], [[-0.8127692329744942, 0.33411022693586073, 0.5364782569010319]]], [[[-0.8102652472572965, -0.3827607641886638, 0.45331894115298943]], [[1.1004085933720784, -0.03340361357901829, 2.242121744944865]], [[0.6627128668525786, 0.45241018086801843, -0.9756918480145411]], [[1.0188219852958569, -0.3339529411388546, -0.1060413360274515]], [[-1.7230119806109447, 1.2166499358694685, 2.0253291042739936]], [[0.19023941246124637, -0.2631192941046416, 0.009234060528456366]]]], "dtype": "float64"}, {"__nd__": [[[[-0.39424163727421924, 0.7371825862139638, 0.5479235187655824]], [[-0.17128316263836424, -1.170484457051967, -0.45680497121283536]], [[-0.11254403363107546, -0.706657753699432, 1.9894134857829946]], [[-1.4611112682437288, -1.1313004143301686, -0.27724472262958766]], [[0.9777094629993583, -0.43345634355240387, -0.614144129428425]], [[0.8273763969123065, 1.7734873521219707, -0.39370214200938736]]], [[[1.016453221819575, 1.1822900591376522, -0.10447782030776247]], [[-0.907336013483252, -0.09201260899109473, -0.09092259406337014]], [[0.876230949025332, -0.5064586620334721, 2.4545530514444414]], [[1.283694975730128, 0.18520927186017466, 0.2661680099090421]], [[-0.19830156098407936, -1.3615657215246473, 0.17520348205600117]], [[0.5832382312310841, -0.09878330744762188, 1.0194769639920829]]]], "dtype": "float64"}, {"__nd__": [[[[0.08968162764764126, -0.004618938602011882, -0.8044217404068754]], [[0.7575094615549641, -1.4217536395401342, -0.282862000345113]], [[-0.35201587444470883, 1.5906082534853276, -1.2235006851315813]], [[0.06274673156214197, 0.9625849495087425, -0.2573077714415755]], [[1.1715499006017833, -0.05631542953606523, 0.31386303657550757]], [[0.9491098944118185, 0.8436272821988037, -0.6524414863699907]]], [[[1.240451844022139, -0.5947184080981095, 1.9393084997453207]], [[-0.5490276324467384, 0.6280160102896578, -0.3623229352273184]], [[1.322131094960442, 1.213219983795988, 0.5001099821433027]], [[0.6564597990225409, -2.2960217763413504, -0.6838918738493461]], [[0.13374635494870832, -0.07209678860114954, 0.09914982355231312]], [[0.5468747838531607, 1.891645031855301, 0.24557435644168232]]]], "dtype": "float64"}, {"__nd__": [[[[0.019911861545683283, 0.2626947110765344, 0.8861006955935505]], [[0.9922116990490443, 0.6402243006445664, 1.67533481596353]], [[0.42673984781208407, 0.3915209242254197, -0.4466836477805744]], [[0.9695386062499501, -0.15950279419102564, 1.5317474785878247]], [[-0.45550123392526237, 0.7535435842968438, 0.19398224322479324]], [[-1.0351978933936161, -3.6838048307738327, 0.3679904915145908]]], [[[-0.2110403017587423, -0.6791875189646711, -1.0061830782506818]], [[0.7952297439718878, -0.05709851123853785, -1.6943903629006443]], [[2.016579893576191, -0.813542048191764, -0.9028094107921949]], [[0.3668245891265635, -1.454307759396053, 0.06019669009366123]], [[1.8078911695606268, 0.23348175268899693, 0.05354737917987486]], [[0.3335701653952812, -0.3694671187034387, -1.788043403375717]]]], "dtype": "float64"}, {"__nd__": [0, 0], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.08968162536621094, -0.00461893854662776, -0.8044217228889465]], [[0.7575094699859619, -1.4217536449432373, -0.28286200761795044]], [[-0.352015882730484, 1.5906082391738892, -1.2235007286071777]], [[0.06274673342704773, 0.9625849723815918, -0.257307767868042]], [[1.171549916267395, -0.056315429508686066, 0.31386303901672363]], [[0.9491099119186401, 0.8436272740364075, -0.6524415016174316]]], [[[1.2404518127441406, -0.5947183966636658, 1.939308524131775]], [[-0.5490276217460632, 0.6280159950256348, -0.36232292652130127]], [[1.3221310377120972, 1.2132200002670288, 0.5001099705696106]], [[0.6564598083496094, -2.2960216999053955, -0.6838918924331665]], [[0.13374635577201843, -0.07209678739309311, 0.09914982318878174]], [[0.5468747615814209, 1.891645073890686, 0.24557435512542725]]]], "dtype": "float32"}, {"__nd__": [[[[0.019911861047148705, 0.26269471645355225, 0.886100709438324]], [[0.9922116994857788, 0.640224277973175, 1.6753348112106323]], [[0.4267398416996002, 0.3915209174156189, -0.4466836452484131]], [[0.9695386290550232, -0.1595027893781662, 1.5317474603652954]], [[-0.4555012285709381, 0.7535435557365417, 0.19398224353790283]], [[-1.0351978540420532, -3.683804750442505, 0.36799049377441406]]], [[[-0.21104030311107635, -0.679187536239624, -1.0061830282211304]], [[0.7952297329902649, -0.05709851160645485, -1.6943904161453247]], [[2.0165798664093018, -0.8135420680046082, -0.9028093814849854]], [[0.36682459712028503, -1.4543077945709229, 0.06019669026136398]], [[1.8078911304473877, 0.2334817498922348, 0.05354737862944603]], [[0.3335701525211334, -0.3694671094417572, -1.7880433797836304]]]], "dtype": "float32"}, {"__nd__": [6, 6], "dtype": "int32"}, {"__nd__": [[true, true, true, true, true, true], [true, true, true, true, true, true]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0173417904689488, 0.4382520350352387], [-0.507529473319981, -0.850798353542014]], [[-0.02238270245778208, -0.11908480127788763], [1.1936131606315683, 0.4226903637080653]], [[0.29272425736180446, -0.3078646240965267], [-1.415810746863587, -0.9162734962608179]], [[0.17592377543291798, 0.5775917622682136], [0.42635270995347735, -1.2703445497049264]]], [[[1.1351510692702491, -0.6876002220831239], [0.49024869891304984, 0.940346308167479]], [[-0.6311093877049172, 1.1790302384998612], [-1.4903804942665038, -0.4058053712483142]], [[-1.1592834820478732, 1.0041768293279087], [0.5340714878417252, 2.355207371659059]], [[1.0047773377064693, 0.2672922016056972], [-1.445702396720866, -0.555157852809296]]]], "dtype": "float64"}, {"__nd__": [[[[-0.007850342542998233, -0.33793689990793674], [1.0148349095897014, -0.4257387388794346]], [[-0.7295595351805871, -0.6381538059813546], [-1.1284088236432894, -2.0572773180262813]], [[0.6418430431142659, 0.812239152353489], [-0.17579807439538883, -0.39252455463903385]], [[-1.9610156874026121, 2.454794045146552], [0.7973786032868776, -0.580100019762124]]], [[[-0.7471664865181128, -0.02948822493462419], [1.7151936469919358, -0.6769058432751488]], [[-0.4907867790348857, -1.2432804224489564], [-0.629370627717362, 1.0614474739900883]], [[-0.022934859663249596, -1.0434047110143614], [2.1684499264289494, -0.730390091863445]], [[0.7883300591874339, 0.8506142382756275], [-1.6846656439345948, -1.5566832899511587]]]], "dtype": "float64"}, {"__nd__": [6, 3], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.017341790720820427, 0.4382520318031311], [-0.5075294971466064, -0.8507983684539795]], [[-0.022382702678442, -0.11908479779958725], [1.1936131715774536, 0.42269036173820496]], [[0.29272425174713135, -0.30786463618278503], [-1.4158107042312622, -0.9162734746932983]], [[0.17592377960681915, 0.5775917768478394], [0.42635270953178406, -1.2703444957733154]]], [[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[1.1351510286331177, -0.6876001954078674], [0.4902487099170685, 0.9403463006019592]], [[-0.6311094164848328, 1.179030179977417], [-1.4903805255889893, -0.405805379152298]], [[-1.1592835187911987, 1.0041768550872803], [0.5340715050697327, 2.3552074432373047]], [[1.0047773122787476, 0.26729220151901245], [-1.4457024335861206, -0.5551578402519226]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[-0.00785034243017435, -0.3379369080066681], [1.0148348808288574, -0.42573875188827515]], [[-0.7295595407485962, -0.6381537914276123], [-1.1284087896347046, -2.057277202606201]], [[0.6418430209159851, 0.8122391700744629], [-0.17579807341098785, -0.3925245404243469]], [[-1.9610157012939453, 2.454793930053711], [0.7973785996437073, -0.5800999999046326]]], [[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[-0.7471665143966675, -0.029488224536180496], [1.7151936292648315, -0.6769058704376221]], [[-0.4907867908477783, -1.2432804107666016], [-0.6293706297874451, 1.0614475011825562]], [[-0.02293485961854458, -1.0434046983718872], [2.168449878692627, -0.7303900718688965]], [[0.788330078125, 0.8506142497062683], [-1.6846656799316406, -1.5566833019256592]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [10, 7], "dtype": "int32"}, {"__nd__": [[true, true, true, true, true, true, true, true, true, true], [true, true, true, true, true, true, true, false, false, false]], "dtype": "bool"}]}}, {"a": [{"__nd__": [[[[0.059778847183861984, -0.7523507048583348], [1.0682337304314435, 0.4110593984143405], [-2.2110419623429007, 1.87486191713291]], [[-1.0874803844966034, -0.29525344958425204], [0.7833071979789376, 0.23767552403200629], [0.0858956881194715, 0.10978343617536218]], [[0.21787050359334145, 0.6732654207470786], [-1.4115744139183604, 0.4243217042785832], [0.04882794031850483, 1.0810247339503454]], [[0.6181220873343874, -1.4660487490061218], [-1.8816366539172589, 0.8096964855316582], [-1.3019734164424648, 0.8386923316927399]], [[1.2863189637022525, -0.050782755115842675], [0.3461547888321997, -2.1933110147698787], [1.1660610147091897, -0.6002394136504806]]], [[[0.9814915306254615, 0.27375654591115545], [0.5458183511940143, -0.7148118260810348], [-0.5284249138332724, -3.6265817033481933]], [[0.33356673184722263, -1.401843159963692], [-0.05653408971598134, -2.1896588365745564], [-0.8324783309528817, -0.8379404906811592]], [[-0.3032761632951546, -0.970287699561292], [-0.11597840692990849, 2.6246159712875037], [-0.3062775504036732, -0.8460131152199474]], [[-0.12266154802320908, 0.08876653974322571], [1.8742351130092305, -0.7897008148716237], [0.29490488591209596, -1.8421791235621394]], [[0.3951086722327658, -0.3137847378152722], [-1.5992772966578808, 1.1317451222568335], [-2.0830350145090066, -0.7383976116639122]]], [[[-1.2577260193078779, -1.0900071292311744], [0.20034273625732954, -2.379171979895629], [0.08361663782854573, 1.125271016870699]], [[-0.7892770034775657, 0.4642095655137767], [0.44198295631808804, 0.4088534767246288], [-0.10015593716766487, 0.10638733681979363]], [[-1.1744532880061558, -0.1712319497816644], [-1.97447314487913, 1.2464711400033506], [-0.007744878387388241, -1.5805652309297489]], [[1.1301675540475127, -0.3951199295913524], [-0.1292955390285947, 0.21661502475934286], [-1.405329389377342, 0.5370659719879886]], [[-0.5929172279474171, 1.5857774441581058], [-2.0524307429961297, 1.7572976409283414], [-0.16169433887208215, 0.7285213549783696]]], [[[-0.9495557793421086, 0.38040445964592456], [0.10765563618678407, 0.5120077799953693], [-0.08969133423034915, 1.5111838787287513]], [[1.3041889077044553, 0.8248404866421202], [0.66690761498961, -0.5511282439572807], [2.3892617033576684, 0.28170493747059094]], [[-0.18347088554468052, 0.2308319318188624], [-1.259479985909002, -0.7252133289548749], [0.9939789860385683, -0.7681834018804485]], [[1.3316776560340724, -0.48385392757282464], [2.3438109346663474, 0.4491232942271467], [-0.2783395055734538, 0.6106049224141757]], [[0.45405509903929314, -0.2636757663774042], [0.39745660177690173, 0.2884057050628073], [-2.5680127884958504, -1.1810615331939174]]]], "dtype": "float64"}, {"__nd__": [[[[0.8792011694904684, -1.0293154428433522], [1.2794146678901437, -0.20569073711668018], [0.020347676940324896, -0.85559186978061]], [[-0.07378487906286636, 1.1090752864868887], [1.8679838079216515, -0.9955932492023629], [0.4562006924718533, -1.5898792020101506]], [[-2.1270225696348977, -1.5253404495081713], [0.018534679538743142, -0.25079959720407724], [0.09893880945206289, 1.159746017323631]], [[0.9929666389898066, -0.12157907908747731], [-0.9543219291478862, -0.593538209027206], [-0.047901875607773925, 0.5378646457323887]], [[-0.14177204340129226, 0.09222246300884764], [-1.7657864382767368, -0.5013553074097618], [-0.25530497467811275, -2.3244581461913847]]], [[[-1.1493669128382966, -0.7188100772240749], [0.048052031003470795, -0.09611150019598871], [-2.103246512114464, 0.027393936746810673]], [[1.37012624314075, 0.7471953642792845], [-0.8326803229732077, -1.1175280306234403], [1.3884712546269342, -0.5607914147158278]], [[-0.6774902661637104, -1.8659295534463516], [1.6880299230119358, -0.2035694140946829], [-1.5117009587087986, -0.5181983860293083]], [[0.07721675784989551, 1.22049418502407], [0.09714349581757505, -0.9296338900166036], [0.4367712410360152, -0.3381598321248262]], [[0.07904093096011056, 0.12488477918168846], [-0.30462920847096236, -0.3353297737641328], [-1.882793140150214, 0.8420508318673523]]], [[[-0.507596582611924, 0.050531762553504125], [0.428141824652351, -0.47270905543216135], [0.7688471799812033, 0.5595378834792586]], [[0.9962739783803941, 0.39252216620205715], [0.8847763355086886, 1.4463323951475135], [-0.6199049114242254, -1.599059442225441]], [[-0.4051728926399283, 0.24274657992595827], [-1.1394300036262206, -0.5609710686082313], [0.655980729941344, 1.3506753034250196]], [[-0.3937039029300722, 0.7127664188849913], [0.4320504669246334, -0.9064494647206303], [-0.25836899406794706, 0.6849913148235863]], [[0.018710534322272286, 0.5099806313640135], [-1.32726284954899, -0.11872261965694882], [-0.8648064170018889, -1.3350370811597527]]], [[[0.10962769315883067, -0.3147867465241862], [-1.4131208993511817, 0.8181933381561604], [-1.753586451928991, -1.0318232995457177]], [[1.0981191840920905, 0.5253598591601333], [0.8351294135791827, -0.47901712667077906], [-0.07358598612304267, -1.881149683463725]], [[-1.214587863343256, 1.1695152410466314], [0.4034479594757982, -0.036028087854260725], [0.7663724381560849, 0.8518468903237236]], [[-1.2019438527100146, 0.041555261750160076], [0.3757281434188403, -1.0869079093753493], [0.4665370382194458, -0.6492030198936337]], [[0.9892491297147336, 0.4328921503478454], [-0.123076892054916, -1.1997851934987336], [0.8181087571481767, -0.04548432316211054]]]], "dtype": "float64"}, {"__nd__": [[[[-0.3375183413299989, -0.6255650946002488], [0.12173629380207801, 0.955908356041404], [0.3467902384312273, 1.0489983618301917]], [[0.6139933811259883, -0.3188766349565948], [0.35684177243550075, 0.24324500037622335], [0.06564228364655991, -0.13354047449681172]]], [[[0.3405504868958516, 0.9996708196294452], [1.1185528400398874, -1.0892182691543784], [0.6961561625950473, 1.284757383348112]], [[-0.4799020116734757, 0.587203029244144], [-0.17185352888006145, -0.3219412660886684], [1.164182110709214, 1.1046168777573662]]], [[[0.11061223524550852, 0.332325417565577], [-1.5865335899261221, -2.1872182714539194], [-1.2055217861208, -1.8188149877395368]], [[-0.8720384223122815, 1.1570832816999321], [1.1385183755775579, 1.6653655896865522], [0.912100268474451, -0.4375051885948231]]], [[[-0.9937634880518732, -6.525737652975222e-05], [2.1840530634712483, -1.733596664133153], [1.1941112161823937, 0.31595884787568174]], [[-1.190316485024396, -0.84547407111019], [-0.6265034049855873, -0.8053488960709784], [-1.5134280464405026, -0.1113062702704002]]]], "dtype": "float64"}, {"__nd__": [[[[0.3854229778972094, -0.8553492433590584], [-0.3793721493964353, -0.9549862147931352], [0.7344905170925535, -0.5124441326600503]], [[-0.22892052902906782, 1.4329358858408168], [0.5034366806930614, 1.8530613011459205], [1.1590497222126357, 0.5037163715232688]]], [[[-0.48020197738225645, 0.3392771986887056], [1.1407054214220962, -0.5956440096280083], [-0.707034374179313, -1.0815564058467457]], [[0.1613927068421146, -0.8573068458425904], [0.16002449491116486, 0.3865390895540662], [0.9525036703034537, 0.1313505903006006]]], [[[0.3989635510012173, -0.7931771332984908], [-0.8189288605523705, 1.5890594967161746], [0.22068872735432132, 0.22839182213593498]], [[1.0304130088774925, -0.7203527381504787], [1.1803574602102838, 1.6987884665428883], [1.5922127973724085, 0.4920646138102318]]], [[[0.2294923983251973, 0.42183515348844286], [-0.13566739407684936, -1.1901588799944838], [0.9337561147554861, -1.604876423629362]], [[-0.4429665381732097, -0.06750415081195307], [-1.306177826327706, -0.21420849396342953], [1.2250519757302756, 0.14287949012681464]]]], "dtype": "float64"}, {"__nd__": [3, 0, 1, 2], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.0597788468003273, -0.7523506879806519], [1.0682337284088135, 0.4110594093799591], [-2.2110419273376465, 1.8748619556427002]], [[-1.0874804258346558, -0.2952534556388855], [0.7833071947097778, 0.2376755177974701], [0.08589568734169006, 0.10978343337774277]], [[0.21787050366401672, 0.6732653975486755], [-1.411574363708496, 0.424321711063385], [0.04882793873548508, 1.0810247659683228]], [[-0.3375183343887329, -0.6255651116371155], [0.12173629552125931, 0.9559083580970764], [0.34679022431373596, 1.0489983558654785]], [[0.6139934062957764, -0.31887662410736084], [0.3568417727947235, 0.24324500560760498], [0.06564228236675262, -0.13354048132896423]]], [[[0.3405504822731018, 0.9996708035469055], [1.118552803993225, -1.089218258857727], [0.6961561441421509, 1.284757375717163]], [[-0.4799019992351532, 0.5872030258178711], [-0.17185352742671967, -0.3219412565231323], [1.1641820669174194, 1.1046168804168701]], [[-0.3032761514186859, -0.9702876806259155], [-0.11597840487957001, 2.6246159076690674], [-0.3062775433063507, -0.8460131287574768]], [[-0.1226615458726883, 0.08876653760671616], [1.8742351531982422, -0.7897008061408997], [0.294904887676239, -1.8421791791915894]], [[0.3951086699962616, -0.31378474831581116], [-1.5992772579193115, 1.1317451000213623], [-2.0830349922180176, -0.7383975982666016]]], [[[-1.2577260732650757, -1.0900071859359741], [0.20034272968769073, -2.3791720867156982], [0.08361663669347763, 1.125270962715149]], [[0.11061223596334457, 0.33232542872428894], [-1.586533546447754, -2.187218189239502], [-1.205521821975708, -1.818814992904663]], [[-0.8720384240150452, 1.15708327293396], [1.1385183334350586, 1.6653655767440796], [0.9121002554893494, -0.43750518560409546]], [[1.1301676034927368, -0.39511993527412415], [-0.12929554283618927, 0.21661502122879028], [-1.4053293466567993, 0.5370659828186035]], [[-0.5929172039031982, 1.5857774019241333], [-2.0524306297302246, 1.7572976350784302], [-0.16169433295726776, 0.7285213470458984]]], [[[-0.9495557546615601, 0.3804044723510742], [0.10765563696622849, 0.5120077729225159], [-0.08969133347272873, 1.5111838579177856]], [[1.3041889667510986, 0.8248404860496521], [0.6669076085090637, -0.5511282682418823], [2.3892617225646973, 0.28170493245124817]], [[-0.9937635064125061, -6.525737990159541e-05], [2.1840531826019287, -1.733596682548523], [1.1941112279891968, 0.3159588575363159]], [[-1.1903164386749268, -0.8454740643501282], [-0.6265034079551697, -0.8053488731384277], [-1.5134280920028687, -0.11130627244710922]], [[0.4540551006793976, -0.2636757791042328], [0.3974566161632538, 0.28840571641921997], [-2.5680127143859863, -1.1810615062713623]]]], "dtype": "float32"}, {"__nd__": [[[[0.8792011737823486, -1.02931547164917], [1.2794146537780762, -0.20569074153900146], [0.020347677171230316, -0.855591893196106]], [[-0.07378488034009933, 1.1090753078460693], [1.8679838180541992, -0.995593249797821], [0.4562006890773773, -1.5898791551589966]], [[-2.1270225048065186, -1.5253404378890991], [0.01853467896580696, -0.25079959630966187], [0.09893880784511566, 1.1597460508346558]], [[0.3854229748249054, -0.8553492426872253], [-0.37937214970588684, -0.9549862146377563], [0.7344905138015747, -0.5124441385269165]], [[-0.22892053425312042, 1.4329358339309692], [0.5034366846084595, 1.853061318397522], [1.1590497493743896, 0.5037163496017456]]], [[[-0.48020198941230774, 0.3392772078514099], [1.1407054662704468, -0.5956439971923828], [-0.7070343494415283, -1.0815564393997192]], [[0.1613927036523819, -0.8573068380355835], [0.160024493932724, 0.386539101600647], [0.9525036811828613, 0.1313505917787552]], [[-0.6774902939796448, -1.8659296035766602], [1.6880298852920532, -0.2035694122314453], [-1.511700987815857, -0.5181983709335327]], [[0.07721675932407379, 1.2204941511154175], [0.0971434935927391, -0.9296339154243469], [0.4367712438106537, -0.33815982937812805]], [[0.07904092967510223, 0.12488477677106857], [-0.30462920665740967, -0.3353297710418701], [-1.8827931880950928, 0.8420508503913879]]], [[[-0.5075966119766235, 0.05053176358342171], [0.42814183235168457, -0.472709059715271], [0.7688471674919128, 0.5595378875732422]], [[0.3989635407924652, -0.7931771278381348], [-0.8189288377761841, 1.5890594720840454], [0.22068873047828674, 0.2283918261528015]], [[1.030413031578064, -0.7203527092933655], [1.1803574562072754, 1.6987885236740112], [1.5922127962112427, 0.49206462502479553]], [[-0.3937039077281952, 0.7127664089202881], [0.4320504665374756, -0.9064494371414185], [-0.25836899876594543, 0.6849913001060486]], [[0.018710535019636154, 0.5099806189537048], [-1.3272628784179688, -0.11872261762619019], [-0.8648064136505127, -1.335037112236023]]], [[[0.10962769389152527, -0.31478673219680786], [-1.4131208658218384, 0.8181933164596558], [-1.7535864114761353, -1.0318232774734497]], [[1.0981191396713257, 0.5253598690032959], [0.83512943983078, -0.47901713848114014], [-0.07358598709106445, -1.8811496496200562]], [[0.2294923961162567, 0.42183515429496765], [-0.13566739857196808, -1.1901588439941406], [0.9337561130523682, -1.6048763990402222]], [[-0.4429665505886078, -0.0675041526556015], [-1.3061778545379639, -0.21420849859714508], [1.225051999092102, 0.14287948608398438]], [[0.9892491102218628, 0.4328921437263489], [-0.123076893389225, -1.1997852325439453], [0.8181087374687195, -0.04548432305455208]]]], "dtype": "float32"}, {"__nd__": [5, 2, 3, 4], "dtype": "int32"}, {"__nd__": [[true, true, true, true, true], [true, true, false, false, false], [true, true, true, false, false], [true, true, true, true, false]], "dtype": "bool"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(update_cache, json.loads(_TESTS), 'jax', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")